# EDA exploratória da base "silver" — escola pública, no censo, com alvo do IDEB

Contexto: no `06_eda_ideb_escola.ipynb` confirmei que dá pra montar uma variável-alvo a partir do `br_inep_ideb/escola`, que o `id_escola` bate de verdade com a `escola_completo`, e tomei quatro decisões: (1) alvo binário `ideb >= 6.0`, (2) só rede pública, (3) só escolas que aparecem na `escola_completo`, e (4) ano de alinhamento = **2025** (decisão final, documentada na Seção 15 do notebook 06: é o ciclo do IDEB com maior preenchimento entre os mais recentes, é posterior ao Censo de 2024 — lógica causal mais defensável —, e faz sentido porque infraestrutura escolar muda pouco de um ano pro outro, então o Censo de 2024 ainda descreve bem a escola em 2025).

**Objetivo deste notebook:** agora que tenho a base já filtrada (a "silver"), fazer uma EDA de verdade em cima das 455 colunas da `escola_completo` como *features* candidatas — separando por grupo temático (infraestrutura, corpo docente, matrícula, etc., já que 455 colunas de uma vez não dá pra olhar direito), vendo preenchimento por grupo, e calculando correlação de cada variável numérica com os dois candidatos a alvo (`ideb` e `taxa_aprovacao`, já que ainda não fechei qual dos dois vai virar a variável-resposta final — ver pendência no `06_eda_ideb_escola.ipynb`).

Não decido nenhuma feature final aqui — isso é levantamento; a seleção de fato acontece quando eu formalizar o pré-processamento em `src/preprocessing`.

In [ ]:
import sys
sys.path.append("../..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 100)

## 1. Montar a base "silver" (escola pública, no censo, com alvo do IDEB)

Reconstruo aqui, a partir do zero, os mesmos filtros já decididos no `06_eda_ideb_escola.ipynb`: escolas que existem na `escola_completo`, só rede pública, recorte de anos iniciais (mais próximo da alfabetização), e o ano definido em `ANO_ESCOLHIDO` (2025, decisão final — fácil de trocar se eu precisar reavaliar mais pra frente, é só mudar essa constante e rodar de novo).

In [ ]:
ANO_ESCOLHIDO = 2025  # decisão final (ver Seção 15 do notebook 06) - infra do Censo 2024 pareada com IDEB de 2025

CACHE_IDEB_ESCOLA = Path("../..") / "data" / "processed" / "ideb_escola.parquet"
CACHE_ESCOLA_COMPLETO = Path("../..") / "data" / "processed" / "escola_completo.parquet"

ideb_escola = pd.read_parquet(CACHE_IDEB_ESCOLA)
escola_completo = pd.read_parquet(CACHE_ESCOLA_COMPLETO)

ideb_escola["id_escola"] = ideb_escola["id_escola"].astype(str)
escola_completo["id_escola"] = escola_completo["id_escola"].astype(str)

interseccao = set(ideb_escola["id_escola"].unique()) & set(escola_completo["id_escola"].unique())

ideb_filtrado = ideb_escola[
    ideb_escola["id_escola"].isin(interseccao)
    & (ideb_escola["rede"].astype(str).str.lower() != "privada")
    & (ideb_escola["anos_escolares"] == "iniciais (1-5)")
    & (ideb_escola["ano"] == ANO_ESCOLHIDO)
].copy()

print(f"Linhas em ideb_filtrado (censo + pública + anos iniciais + ano {ANO_ESCOLHIDO}): {len(ideb_filtrado):,}")

duplicadas = ideb_filtrado["id_escola"].duplicated().sum()
if duplicadas:
    print(f"Atenção: {duplicadas} id_escola duplicados nesse recorte - mantendo a primeira ocorrência.")
    ideb_filtrado = ideb_filtrado.drop_duplicates(subset="id_escola", keep="first")

colunas_alvo = ["id_escola", "id_municipio", "sigla_uf", "rede", "taxa_aprovacao", "ideb"]
base_eda = ideb_filtrado[colunas_alvo].merge(escola_completo, on="id_escola", how="left", suffixes=("", "_censo"))

print(f"\nbase_eda final: {base_eda.shape[0]:,} escolas, {base_eda.shape[1]} colunas")
print(f"% com 'ideb' preenchido: {base_eda['ideb'].notna().mean() * 100:.2f}%")
print(f"% com 'taxa_aprovacao' preenchido: {base_eda['taxa_aprovacao'].notna().mean() * 100:.2f}%")
base_eda.head()

## 2. Classificação temática das 455 colunas

455 colunas de uma vez não dá pra olhar com cuidado, então separo por grupo temático usando palavras-chave no nome da coluna (o mesmo tipo de abordagem que já usei pra achar candidatas a variável-alvo no `04_eda_escola_completo.ipynb`). Colunas que não batem com nenhum grupo ficam numa categoria "não classificada" pra eu revisar manualmente depois.

In [ ]:
GRUPOS_TEMATICOS = {
    "infraestrutura": [
        "agua", "energia", "esgoto", "internet", "banda_larga", "biblioteca", "laboratorio",
        "quadra", "sala_", "dependencia", "acessibilidade", "cozinha", "refeitorio", "auditorio",
        "parque_infantil", "patio", "almoxarifado", "sanitario", "lixo", "energia_inexistente",
        "dependencia_lixo", "recurso", "equipamento", "banheiro", "piscina", "area_verde",
        "secretaria", "despensa", "terreirao", "viveiro", "predio_compartilhado",
        "local_funcionamento", "tipo_rede_local", "material_pedagogico",
    ],
    "corpo_docente": ["docente", "professor"],
    "corpo_tecnico_administrativo": [
        "profissional_administrativo", "profissional_servico_geral", "profissional_bibliotecario",
        "profissional_saude", "profissional_coordenador", "profissional_psicologo",
        "profissional_nutricionista", "profissional_pedagogia", "profissional_secretario",
        "profissional_seguranca", "profissional_monitor", "profissional_assistente_social",
        "profissional_tradutor", "profissional_fonaudiologo", "profissional_alimentacao",
        "quantidade_profissional",
    ],
    "quantidade_matricula": ["matricula", "aluno", "estudante"],
    "localizacao": ["zona", "localizacao", "localidade", "endereco", "distrito", "regiao"],
    "identificacao": ["id_", "co_", "nome", "entidade", "cnpj", "cep", "sigla_uf", "ano"],
    "gestao_administrativa": ["dependencia_administrativa", "convenio", "vinculo", "regulamentacao",
                              "categoria", "conveniada", "orgao"],
    "modalidade_etapa": ["etapa", "modalidade", "ensino_", "educacao_", "curso", "eja", "creche",
                          "pre_escola", "fundamental", "medio", "profissionalizante"],
    "programa": ["programa"],
}

colunas_ja_usadas = {"id_escola", "id_municipio", "sigla_uf", "rede", "taxa_aprovacao", "ideb"}
colunas_a_classificar = [c for c in base_eda.columns if c not in colunas_ja_usadas]

classificacao = {}
for coluna in colunas_a_classificar:
    coluna_lower = coluna.lower()
    grupo_encontrado = None
    for grupo, palavras in GRUPOS_TEMATICOS.items():
        if any(p in coluna_lower for p in palavras):
            grupo_encontrado = grupo
            break
    classificacao[coluna] = grupo_encontrado or "nao_classificada"

resumo_classificacao = pd.Series(classificacao).value_counts()
print("Colunas por grupo temático:")
print(resumo_classificacao)

print("\nColunas 'nao_classificada' (revisar manualmente, uma a uma):")
nao_classificadas = [c for c, g in classificacao.items() if g == "nao_classificada"]
for c in nao_classificadas:
    print(" -", c)
print(f"... total: {len(nao_classificadas)} colunas não classificadas")

## 3. Preenchimento por grupo temático

Antes de olhar correlação, preciso saber o quanto cada grupo está preenchido nesta base já filtrada (455 colunas, mas agora só nas ~escolas do recorte) — um grupo interessante mas com preenchimento muito ruim não serve pra muita coisa.

In [ ]:
linhas_resumo = []
for coluna, grupo in classificacao.items():
    linhas_resumo.append({
        "grupo": grupo,
        "coluna": coluna,
        "dtype": str(base_eda[coluna].dtype),
        "pct_preenchido": round(base_eda[coluna].notna().mean() * 100, 2),
    })

resumo_preenchimento = pd.DataFrame(linhas_resumo).sort_values(["grupo", "pct_preenchido"], ascending=[True, False])

print("Preenchimento médio por grupo:")
print(resumo_preenchimento.groupby("grupo")["pct_preenchido"].agg(["mean", "min", "max", "count"]).round(2))

print("\nDetalhe completo (coluna a coluna):")
print(resumo_preenchimento.to_string(index=False))

## 4. Distribuição do alvo nesta base: `ideb` e `taxa_aprovacao`

Antes de correlacionar qualquer coisa, olho a distribuição dos dois candidatos a variável-alvo já dentro deste recorte específico (censo + pública + anos iniciais + ano escolhido) — pra confirmar que continuam com cara de dado real, e não um artefato do filtro.

In [ ]:
print("Distribuição de 'ideb' na base_eda:")
print(base_eda["ideb"].describe())

print("\nDistribuição de 'taxa_aprovacao' na base_eda:")
print(base_eda["taxa_aprovacao"].describe())

fig, eixos = plt.subplots(1, 2, figsize=(12, 4))
base_eda["ideb"].dropna().hist(bins=30, ax=eixos[0])
eixos[0].set_title("Distribuição do IDEB")
eixos[0].axvline(6.0, color="red", linestyle="--", label="corte do alvo (6.0)")
eixos[0].legend()

base_eda["taxa_aprovacao"].dropna().hist(bins=30, ax=eixos[1])
eixos[1].set_title("Distribuição da taxa de aprovação")
plt.tight_layout()
plt.show()

## 5. Correlação de cada variável numérica com `ideb` e `taxa_aprovacao`

Uso correlação de Spearman (baseada em ranking, não assume relação linear nem distribuição normal) — mais robusta pra essas colunas de contagem/proporção, que costumam ser bem enviesadas. Calculo contra os dois candidatos a alvo, já que ainda não decidi qual vai ser o definitivo (pendência registrada no `06_eda_ideb_escola.ipynb`).

In [ ]:
colunas_numericas = [
    c for c in base_eda.columns
    if c not in colunas_ja_usadas and pd.api.types.is_numeric_dtype(base_eda[c])
]

linhas_correlacao = []
for coluna in colunas_numericas:
    serie = base_eda[coluna]
    if serie.notna().sum() < 30:
        continue  # preenchimento baixo demais pra uma correlação minimamente confiável
    corr_ideb = base_eda[[coluna, "ideb"]].dropna().corr(method="spearman").iloc[0, 1]
    corr_taxa = base_eda[[coluna, "taxa_aprovacao"]].dropna().corr(method="spearman").iloc[0, 1]
    linhas_correlacao.append({
        "grupo": classificacao.get(coluna, "nao_classificada"),
        "coluna": coluna,
        "pct_preenchido": round(serie.notna().mean() * 100, 2),
        "corr_ideb": round(corr_ideb, 3) if pd.notna(corr_ideb) else np.nan,
        "corr_taxa_aprovacao": round(corr_taxa, 3) if pd.notna(corr_taxa) else np.nan,
    })

tabela_correlacao = pd.DataFrame(linhas_correlacao)
tabela_correlacao["abs_corr_ideb"] = tabela_correlacao["corr_ideb"].abs()

print(f"Total de colunas numéricas avaliadas: {len(tabela_correlacao)}")
print("\n=== Top 30 por correlação absoluta com 'ideb' ===")
print(tabela_correlacao.sort_values("abs_corr_ideb", ascending=False).head(30).to_string(index=False))

In [ ]:
print("=== Top 5 de cada grupo temático, por correlação absoluta com 'ideb' ===")
for grupo, sub in tabela_correlacao.groupby("grupo"):
    print(f"\n--- {grupo} ---")
    print(sub.sort_values("abs_corr_ideb", ascending=False).head(5).to_string(index=False))

## 6. Visualizando as variáveis mais correlacionadas

Um gráfico de barras com as top 15 (por correlação absoluta com `ideb`), pra ver rapidamente quais grupos temáticos dominam o topo — e um scatter/boxplot das 3 mais correlacionadas, pra confirmar visualmente que a correlação numérica não é um artefato (ex.: outliers puxando o número).

In [ ]:
top_15 = tabela_correlacao.sort_values("abs_corr_ideb", ascending=False).head(15)

plt.figure(figsize=(9, 6))
cores = ["#2c7fb8" if v >= 0 else "#d95f0e" for v in top_15["corr_ideb"]]
plt.barh(top_15["coluna"], top_15["corr_ideb"], color=cores)
plt.xlabel("Correlação de Spearman com 'ideb'")
plt.title("Top 15 variáveis mais correlacionadas com o IDEB")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
top_3_colunas = top_15["coluna"].head(3).tolist()

fig, eixos = plt.subplots(1, len(top_3_colunas), figsize=(5 * len(top_3_colunas), 4))
if len(top_3_colunas) == 1:
    eixos = [eixos]

for eixo, coluna in zip(eixos, top_3_colunas):
    dados = base_eda[[coluna, "ideb"]].dropna()
    eixo.scatter(dados[coluna], dados["ideb"], alpha=0.2, s=8)
    eixo.set_xlabel(coluna)
    eixo.set_ylabel("ideb")
    eixo.set_title(coluna)

plt.tight_layout()
plt.show()

## 6.1 Comparando lado a lado: top 10 correlacionadas com `ideb` vs. top 10 com `taxa_aprovacao`

Como ainda não decidi qual dos dois vai ser o alvo final, quero ver se as variáveis mais fortes são as mesmas nos dois casos, ou se cada métrica "escuta" um sinal diferente da escola — isso também ajuda na decisão de qual usar.

In [ ]:
tabela_correlacao["abs_corr_taxa"] = tabela_correlacao["corr_taxa_aprovacao"].abs()

top_10_ideb = tabela_correlacao.sort_values("abs_corr_ideb", ascending=False).head(10)
top_10_taxa = tabela_correlacao.sort_values("abs_corr_taxa", ascending=False).head(10)

fig, (eixo_ideb, eixo_taxa) = plt.subplots(1, 2, figsize=(16, 6))

cores_ideb = ["#2c7fb8" if v >= 0 else "#d95f0e" for v in top_10_ideb["corr_ideb"]]
eixo_ideb.barh(top_10_ideb["coluna"], top_10_ideb["corr_ideb"], color=cores_ideb)
eixo_ideb.set_title("Top 10 correlacionadas com 'ideb'")
eixo_ideb.set_xlabel("Correlação de Spearman")
eixo_ideb.invert_yaxis()

cores_taxa = ["#2c7fb8" if v >= 0 else "#d95f0e" for v in top_10_taxa["corr_taxa_aprovacao"]]
eixo_taxa.barh(top_10_taxa["coluna"], top_10_taxa["corr_taxa_aprovacao"], color=cores_taxa)
eixo_taxa.set_title("Top 10 correlacionadas com 'taxa_aprovacao'")
eixo_taxa.set_xlabel("Correlação de Spearman")
eixo_taxa.invert_yaxis()

plt.tight_layout()
plt.show()

colunas_em_comum = set(top_10_ideb["coluna"]) & set(top_10_taxa["coluna"])
print(f"Colunas que aparecem no top 10 dos dois: {len(colunas_em_comum)}")
for c in colunas_em_comum:
    print(" -", c)

**Achado que merece atenção especial:** `quantidade_matricula_branca` (contagem de matrículas de alunos que se autodeclaram brancos) apareceu como a variável de maior correlação absoluta com `ideb` (0,403) de toda a base — mais forte que qualquer variável de infraestrutura. Isso não é uma relação causal (a cor da pele do aluno não "causa" desempenho), e sim reflexo de um problema bem documentado no Brasil: composição racial da escola correlaciona com contexto socioeconômico e histórico de desigualdade regional/urbana, que por sua vez afeta desempenho. Uso essa variável aqui só pra descrever o padrão (é isso que a EDA é pra fazer). Incluir ou não como feature direta do modelo é uma decisão sensível: usá-la sem tratamento pode fazer o modelo aprender um proxy racial em vez de um fator realmente acionável por política pública (como infraestrutura ou corpo docente). Registro esse ponto aqui como um cuidado ético explícito do projeto, a ser resolvido quando eu fechar a lista final de features em `src/preprocessing` — não é uma decisão que tomo dentro da EDA.

## 7. Variáveis categóricas — distribuição e associação com o alvo

Até aqui só olhei variáveis numéricas (contagens, correlação). Mas a `escola_completo` tem bastante coluna categórica/código (`sigla_uf`, `tipo_localizacao`, `tipo_situacao_funcionamento`, `tipo_regulamentacao`, etc.) que a correlação de Spearman não captura bem quando são poucas categorias sem ordem natural. Identifico aqui as colunas de baixa cardinalidade (poucos valores distintos — jeito prático de achar "categóricas" mesmo em colunas que vieram como número/código) e, pra cada uma, olho a distribuição de escolas por categoria e a taxa de escolas "boas" (uso o corte já decidido, `ideb >= 6.0`, só para esta EDA — não é a implementação final do alvo).

**Nota:** algumas dessas colunas são códigos numéricos (ex.: `tipo_localizacao` provavelmente 1=Urbana, 2=Rural, seguindo o padrão do INEP) sem que eu tenha, neste notebook, a tabela de-para oficial pra traduzir o código em texto — a `br_inep_avaliacao_alfabetizacao.dicionario` (mencionada no catálogo de dados) pode servir pra isso, mas ainda não busquei especificamente pra `escola_completo`. Fica como próximo passo se essas variáveis avançarem para a seleção final.

In [ ]:
base_eda["alvo_binario_ideb"] = (base_eda["ideb"] >= 6.0).astype("Int64")
base_eda.loc[base_eda["ideb"].isna(), "alvo_binario_ideb"] = pd.NA

colunas_candidatas_categoricas = [
    c for c in base_eda.columns
    if c not in colunas_ja_usadas | {"alvo_binario_ideb"}
    and base_eda[c].nunique(dropna=True) <= 15
    and base_eda[c].nunique(dropna=True) >= 2
]

print(f"Colunas de baixa cardinalidade encontradas (candidatas a categóricas): {len(colunas_candidatas_categoricas)}")

# Foco nas mais interessantes pra essa EDA (as outras são flags binárias de infraestrutura,
# já cobertas na correlação da Seção 5) - variáveis de identificação/localização/organização.
colunas_categoricas_foco = [
    c for c in ["rede", "sigla_uf", "tipo_localizacao", "tipo_localizacao_diferenciada",
                "tipo_situacao_funcionamento", "tipo_regulamentacao",
                "tipo_responsavel_regulamentacao"]
    if c in base_eda.columns
]

for coluna in colunas_categoricas_foco:
    print(f"\n=== {coluna} ===")
    resumo = base_eda.groupby(coluna, dropna=False).agg(
        escolas=("id_escola", "size"),
        pct_ideb_preenchido=("ideb", lambda s: round(s.notna().mean() * 100, 2)),
        ideb_medio=("ideb", "mean"),
        pct_acima_da_meta=("alvo_binario_ideb", lambda s: round(s.mean(skipna=True) * 100, 2) if s.notna().any() else np.nan),
    ).round(2).sort_values("escolas", ascending=False)
    print(resumo.to_string())

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(15, 5))

resumo_uf = (
    base_eda.groupby("sigla_uf")["alvo_binario_ideb"]
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)
eixos[0].bar(resumo_uf.index, resumo_uf.values, color="#2c7fb8")
eixos[0].set_title("% de escolas com ideb >= 6.0, por UF")
eixos[0].set_ylabel("% acima da meta")
eixos[0].tick_params(axis="x", rotation=90)

if "tipo_localizacao" in base_eda.columns:
    resumo_localizacao = (
        base_eda.groupby("tipo_localizacao")["alvo_binario_ideb"]
        .mean()
        .mul(100)
        .round(2)
    )
    eixos[1].bar(resumo_localizacao.index.astype(str), resumo_localizacao.values, color="#41ab5d")
    eixos[1].set_title("% de escolas com ideb >= 6.0, por 'tipo_localizacao'\n(código não traduzido - ver nota da Seção 7)")
    eixos[1].set_ylabel("% acima da meta")

plt.tight_layout()
plt.show()

## 7.1 Quantas colunas são numéricas e quantas são categóricas, afinal?

Fiquei em dúvida sobre isso ao revisar a Seção 7: o dtype que o pandas atribui (int/float vs. object) não é a mesma coisa que "é uma variável numérica de verdade" — várias colunas vêm como número (int) mas são, na prática, um código categórico (ex.: `tipo_localizacao`, `rede`), não uma contagem ou proporção com significado de escala. Então faço a contagem em duas camadas: primeiro pelo dtype bruto, depois cruzando com a heurística de cardinalidade que já usei na Seção 7 (`nunique(dropna=True) <= 15`) pra separar, dentro das numéricas, o que é número "de verdade" do que é categoria disfarçada de número.

In [ ]:
dtypes_resumo = base_eda.dtypes.value_counts()
print("Colunas por dtype bruto (como o pandas leu):")
print(dtypes_resumo)

colunas_numericas_dtype = base_eda.select_dtypes(include=[np.number]).columns.tolist()
colunas_objeto_dtype = base_eda.select_dtypes(include=["object"]).columns.tolist()

print(f"\nTotal de colunas na base: {base_eda.shape[1]}")
print(f"Colunas com dtype numérico (int/float): {len(colunas_numericas_dtype)}")
print(f"Colunas com dtype texto/object: {len(colunas_objeto_dtype)}")

# dtype sozinho não conta a história toda - cruzo com a cardinalidade (mesmo corte
# já usado na Seção 7) pra separar, dentro das numéricas, contagens/proporções de
# verdade de códigos categóricos que só parecem número.
numericas_baixa_cardinalidade = [
    c for c in colunas_numericas_dtype
    if base_eda[c].nunique(dropna=True) <= 15
]
numericas_alta_cardinalidade = [c for c in colunas_numericas_dtype if c not in numericas_baixa_cardinalidade]

print(f"\nDentro das colunas numéricas (dtype):")
print(f"  - alta cardinalidade (números de verdade - contagens, proporções, notas): {len(numericas_alta_cardinalidade)}")
print(f"  - baixa cardinalidade (provavelmente código categórico guardado como número): {len(numericas_baixa_cardinalidade)}")

print(f"\nResumo funcional (o que interessa pra decidir encoding na modelagem):")
print(f"  - Categóricas (baixa cardinalidade - vão precisar de encoding): {len(colunas_candidatas_categoricas)}")
print(f"  - Numéricas de verdade (alta cardinalidade - vão para imputação/escala numérica): {base_eda.shape[1] - len(colunas_candidatas_categoricas)}")


## 8. Conclusão preliminar

- **Base montada (Seção 1):** com `ANO_ESCOLHIDO = 2025` (decisão final), 63.537 escolas, 347 colunas — 66,53% com `ideb` preenchido e 68,00% com `taxa_aprovacao` preenchido (mais alto que a base completa do notebook 06 porque aqui já está pré-filtrado pra pública + censo + anos iniciais).
- **Classificação temática (Seção 2):** 8 grupos (infraestrutura, corpo docente, corpo técnico/administrativo, quantidade_matricula, localização, identificação, gestão administrativa, modalidade/etapa, programa) — depois de refinar as palavras-chave, restam bem menos colunas "não classificadas" que na primeira versão; as que sobraram valem uma checagem manual pontual.
- **Preenchimento por grupo (Seção 3):** todos os grupos ficaram bem preenchidos (~96-98%) neste recorte já filtrado — não é um problema para seleção de features.
- **Distribuição do alvo neste recorte (Seção 4):** `ideb` média 6,06 (mediana 6,1) — mais alto que a média geral do notebook 06 (4,67-5,06), porque aqui já filtramos pra escolas com dado completo o suficiente pra aparecer nesse recorte (viés de seleção a documentar: escolas sem SAEB aplicado tendem a ser as menores/mais isoladas, que também tendem a ter IDEB mais baixo quando aparecem).
- **Top correlações com `ideb` (Seção 5):** `quantidade_matricula_branca` lidera (0,403) — acompanhado da ressalva ética registrada acima. Depois vêm variáveis de infraestrutura (TV, internet+computador, tratamento de lixo, quadra de esportes) e uma de gestão (`orgao_associacao_pais_mestres`). `quantidade_matricula_idade_15_17`/`idade_18` (matrícula fora da idade esperada pros anos iniciais — indício de distorção idade-série) correlaciona negativamente, como esperado.
- **`ideb` vs. `taxa_aprovacao` (Seção 6.1):** essa comparação lado a lado é uma célula que adicionei depois da última execução completa do notebook — ainda preciso rodar de novo pra ver se o top 10 de cada alvo se sobrepõe ou aponta pra sinais diferentes. Fica como próximo passo antes de fechar qual dos dois vira a variável-resposta definitiva.
- **Variáveis categóricas (Seção 7):** também é seção nova, ainda sem execução registrada — a ideia é ver se `sigla_uf`/`tipo_localizacao`/`tipo_regulamentacao` mostram diferenças relevantes de `% acima da meta`; olho o resultado assim que rodar o notebook de novo com essas células.

**Isso ainda não é seleção final de features — é levantamento.** Pontos que preciso resolver antes de escrever qualquer código em `src/preprocessing`: (1) o uso ou não de `quantidade_matricula_branca` como feature (questão ética/de proxy, registrada acima); (2) como tratar as colunas de matrícula pra virar "tamanho da escola"; (3) se o alvo final é `ideb`, `taxa_aprovacao` ou uma combinação; (4) o viés de seleção nas escolas com IDEB preenchido, que pode inflar a nota média observada nesta EDA em relação à população real de escolas.